In [59]:
import pandas as pd
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error , r2_score , root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_train = pd.read_csv(r'..\data\processed\train_encoded.csv')
df_train

,building_construction_year,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,country,location,price_in_USD
0,2005.0,6.0,5.0,4.0,3.0,1.0,130.0,110.5,514842.470625,435879.722356,12.690879
1,2006.5,4.5,5.0,4.0,4.0,2.5,173.0,105.0,691462.950309,293534.872239,13.404254
2,2018.0,5.0,1.0,4.0,3.0,2.0,101.0,60.0,691462.950309,437700.428393,12.542027
3,2022.0,8.0,2.0,3.0,2.0,1.5,112.0,61.0,691462.950309,257107.015891,12.858736
4,1986.0,9.0,9.0,2.5,2.0,1.5,65.0,26.0,71635.572400,42734.758045,9.903538
...,...,...,...,...,...,...,...,...,...,...,...
108043,2021.0,3.0,5.5,3.5,4.0,1.0,200.0,172.5,202523.475311,191391.090535,13.328616
108044,2012.0,2.0,2.5,3.0,2.0,1.0,56.0,36.0,278704.759588,342089.984127,12.867053
108045,1978.0,1.0,1.0,1.0,4.0,2.5,255.0,142.0,604180.865107,320406.576660,12.613419
108046,1978.5,11.0,3.0,2.0,1.0,1.0,70.0,64.5,374312.309839,231236.367466,11.953964


In [3]:
df_test = pd.read_csv(r'..\data\processed\test_encoded.csv')
df_test

,building_construction_year,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,country,location,price_in_USD
0,2023.0,8.0,3.0,4.0,4.0,1.0,182.0,153.682146,3.743123e+05,4.375187e+05,13.077887
1,2007.5,7.0,2.5,3.0,3.0,2.0,121.0,59.000000,5.148425e+05,1.051283e+06,12.970000
2,1996.5,6.5,6.0,3.0,3.0,1.5,106.0,45.500000,6.914630e+05,5.134686e+05,12.691559
3,2022.0,12.0,7.0,3.0,2.0,1.0,91.0,91.000000,9.512471e+04,1.701686e+05,12.072547
4,1994.0,3.5,1.0,2.5,3.0,1.5,172.0,47.000000,7.163557e+04,1.050603e+05,11.373675
...,...,...,...,...,...,...,...,...,...,...,...
27007,2023.0,4.0,3.0,2.0,2.0,1.0,51.0,29.000000,3.743123e+05,2.312364e+05,11.753335
27008,2024.0,8.0,2.0,2.0,1.0,1.0,52.0,31.000000,3.743123e+05,1.864179e+05,11.414099
27009,2025.0,6.0,6.0,2.0,2.0,1.0,81.0,42.500000,1.284470e+06,1.346302e+06,12.911645
27010,2022.5,3.5,6.5,2.0,1.0,1.0,70.0,53.500000,6.555824e+05,3.983064e+05,11.778607


In [7]:
target = "price_in_USD"

X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]

In [27]:
import optuna
def objective_lgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 1000, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.03),

        "num_leaves": trial.suggest_int("num_leaves", 16, 64),
        "max_depth": trial.suggest_int("max_depth", 4, 8),

        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),

        "reg_lambda": trial.suggest_float("reg_lambda", 5, 30),
        "reg_alpha": trial.suggest_float("reg_alpha", 3, 15),

        "min_child_samples": trial.suggest_int("min_child_samples", 20, 60),

        "random_state": 42,
        "objective": "regression",
    }

    model = LGBMRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="rmse"
    )

    preds = model.predict(X_test)
    return r2_score(y_test, preds)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective_lgb, n_trials=100)

print("Best R2:", study.best_value)
print("Best Params:", study.best_params)

In [ ]:
best_params = study.best_params
best_params["objective"] = "regression"
best_params["random_state"] = 42

best_lgb = LGBMRegressor(**best_params)

best_lgb.fit(X_train, y_train)

In [ ]:
y_train_pred_log = best_lgb.predict(X_train)
y_test_pred_log  = best_lgb.predict(X_test)

mse_train = mean_squared_error(y_train, y_train_pred_log)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred_log)

mse_test = mean_squared_error(y_test, y_test_pred_log)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred_log)

print("===== LightGBM TUNED Train =====")
print("Train MSE :", mse_train)
print("Train RMSE:", rmse_train)
print("Train R²  :", r2_train)

print("\n===== LightGBM TUNED Test =====")
print("Test MSE :", mse_test)
print("Test RMSE:", rmse_test)
print("Test R²  :", r2_test)

===== LightGBM TUNED Train =====
Train MSE : 0.16511077574949234
Train RMSE: 0.40633825287498143
Train R²  : 0.8638019308873839

===== LightGBM TUNED Test =====
Test MSE : 0.25731234171048617
Test RMSE: 0.5072596393470371
Test R²  : 0.7924282508741777


In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 20),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 50),
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    preds = model.predict(X_test)
    return r2_score(y_test, preds)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective_xgb, n_trials=40)

print("Best Test R2:", study.best_value)
print("Best Parameters:", study.best_params)

In [54]:
best_params = study.best_params
best_params["objective"] = "reg:squarederror"
best_params["tree_method"] = "hist"
best_params["random_state"] = 42

best_xgb = XGBRegressor(**best_params)

best_xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8940896559662486
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

In [ ]:
y_train_pred_log = best_xgb.predict(X_train)
y_test_pred_log  = best_xgb.predict(X_test)


mse_train = mean_squared_error(y_train, y_train_pred_log)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred_log)

mse_test = mean_squared_error(y_test, y_test_pred_log)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred_log)

print("===== XGBoost TUNED Train =====")
print("Train MSE :", mse_train)
print("Train RMSE:", rmse_train)
print("Train R²  :", r2_train)

print("\n===== XGBoost TUNED Test =====")
print("Test MSE :", mse_test)
print("Test RMSE:", rmse_test)
print("Test R²  :", r2_test)

===== XGBoost TUNED Train =====
Train MSE : 0.1470214538681161
Train RMSE: 0.38343376725076794
Train R²  : 0.8787236142276525

===== XGBoost TUNED Test =====
Test MSE : 0.26600148367961635
Test RMSE: 0.5157533166927929
Test R²  : 0.7854187915340413


In [64]:
import joblib
joblib.dump(best_lgb, "../artifacts/final_model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [67]:
import json
feature_list = list(X_train.columns)
with open("../artifacts/features.json", "w") as f:
    json.dump(feature_list, f)
print("Saved feature list!")

Saved feature list!
